# Betting backtest: Premier League 2023/24 holdout

Samma modell och odds som i `betting_backtest.ipynb`, men med upplägget från `premier_league_all_history.ipynb`:

- Träna på **all Premier League-historik före 2023/24**.
- 2023/24 är en helt orörd testsäsong.
- ROI mäts mot Pinnacle closing på den säsongen.
- Samma sak igen, men bara när modellen är mer säker. Säkerhetsgränsen är medianen av modellens högsta H/D/A-sannolikhet i historisk walk-forward (2018/19–2022/23), inte på teståret.
- Därefter: säkra matcher där **marknaden också pickar samma utfall**, men **modellen ger det högre sannolikhet** än den deviggade closing-linjen.
- Avsnitt 5–7 settlar mot **Pinnacle closing**. Samma urval, men till **högsta namngivna bolagsodds**, kommer i avsnitt 11.

Ett vad är 1 enhet på modellens mest sannolika utfall, till det pris bookmakern faktiskt erbjöd. Kontrollen är ett slumpmässigt utfall per match.

In [61]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

here = Path.cwd().resolve()
root = next(path for path in [here, *here.parents] if (path / "src" / "football_data.py").exists())
sys.path.insert(0, str(root))

from src import backtest
from src.feature_engineering import engineer_features
from src.football_data import load_matches
from src.team_dataset import build_team_match_dataset
from src.team_model import (
    BEST_LOGISTIC,
    fit_best_logistic,
    model_input_columns,
    probabilities_in_order,
    split_train_test,
)

LEAGUE = "Premier League"
TEST_SEASON = "2023/24"
CV_FIRST_SEASON = "2018/19"
OUTCOMES = backtest.OUTCOMES
EDGE_THRESHOLD = 0.05
MIN_PROB_EDGE = 0.0
CONFIDENCE_SWEEP = (0.40, 0.45, 0.50, 0.55, 0.60, 0.70)
PROB_EDGE_SWEEP = (0.00, 0.01, 0.02, 0.03, 0.05, 0.08)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("pandas:", pd.__version__)
print("Modell:", BEST_LOGISTIC)
print("Test-säsong:", TEST_SEASON)
print("OOF-säkerhet från:", CV_FIRST_SEASON, "→ säsongen före", TEST_SEASON)

pandas: 1.5.3
Modell: {'feature_set': 'form_strength_model', 'C': 0.03, 'class_weight': None}
Test-säsong: 2023/24
OOF-säkerhet från: 2018/19 → säsongen före 2023/24


## 1. Ladda Premier League med odds

Historiken klipps vid testsäsongen så senare matcher inte kan påverka features. Odds kommer från football-data.co.uk, inte spegeln bakom `data_loader`.

In [62]:
raw_all = load_matches(divisions=["E0"])
raw = raw_all.loc[
    (raw_all["League"] == LEAGUE) & (raw_all["Season"].astype(str) <= TEST_SEASON)
].copy()
raw["MatchDate"] = pd.to_datetime(raw["MatchDate"])
raw = raw.sort_values(["MatchDate", "HomeTeam", "AwayTeam"]).reset_index(drop=True)
if raw.empty:
    raise ValueError("Inga matcher laddades.")

closing = backtest.CLOSING_ODDS
has_closing = raw[closing].notna().all(axis=1) & (raw[closing] > 1).all(axis=1)
test_raw = raw.loc[raw["Season"].astype(str).eq(TEST_SEASON)]

print(f"Matcher t.o.m. {TEST_SEASON}: {len(raw):,}")
print(f"Train-säsonger: {raw.loc[raw.Season.astype(str) < TEST_SEASON, 'Season'].min()}–{raw.loc[raw.Season.astype(str) < TEST_SEASON, 'Season'].max()}")
print(f"Test-säsong: {TEST_SEASON}, {len(test_raw)} matcher")
print(f"Pinnacle closing på test: {int(has_closing.loc[test_raw.index].sum())} / {len(test_raw)}")
test_raw.loc[has_closing.reindex(test_raw.index, fill_value=False), [
    "MatchDate", "HomeTeam", "AwayTeam", "FullTimeResult", *closing,
]].head()

Matcher t.o.m. 2023/24: 9,120
Train-säsonger: 2000/01–2022/23
Test-säsong: 2023/24, 380 matcher
Pinnacle closing på test: 380 / 380


,MatchDate,HomeTeam,AwayTeam,FullTimeResult,PSCH,PSCD,PSCA
8740,2023-08-11,Burnley,Man City,A,9.6200,5.8100,1.3300
8741,2023-08-12,Arsenal,Nott'm Forest,H,1.1900,8.0000,16.0000
8742,2023-08-12,Bournemouth,West Ham,D,2.7500,3.6000,2.6300
8743,2023-08-12,Brighton,Luton,H,1.2700,6.3600,11.3600
8744,2023-08-12,Everton,Fulham,A,2.3900,3.3200,3.3000


## 2. Features, lagdata och holdout-split

Varje match blir två lagrader. Modellen tränas bara på säsonger före 2023/24. Features för en testmatch får använda tidigare matcher i testsäsongen, samma regel som i all-history-notebooken.

In [63]:
print("Bygger features…")
engineered = engineer_features(raw)
team_data = build_team_match_dataset(raw, engineered)
train_data, test_data = split_train_test(team_data, test_season=TEST_SEASON)

assert set(train_data["MatchId"]).isdisjoint(test_data["MatchId"])
print(f"Train: {len(train_data):,} lagrader, {train_data.Season.nunique()} säsonger")
print(f"Test:  {len(test_data):,} lagrader, {test_data.Season.nunique()} säsong")

Bygger features…
Train: 17,480 lagrader, 23 säsonger
Test:  760 lagrader, 1 säsong


## 3. Träna på historiken, prediktera testsäsongen

Samma logistiska modell som i all-history (`form_strength_model`, `C=0.03`). De två lagraderna slås ihop till en H/D/A-fördelning per match och joinas mot Pinnacle closing.

In [64]:
model = fit_best_logistic(train_data)
test_probs = probabilities_in_order(model, test_data, model_input_columns())
predictions = backtest.match_probabilities(test_data, test_probs)
predictions["Season"] = TEST_SEASON
predictions["confidence"] = predictions[OUTCOMES].max(axis=1)
predictions["pick"] = predictions[OUTCOMES].idxmax(axis=1)

market = backtest.build_market(raw)
joined = predictions.join(market, how="inner")
print(f"Testmatcher med prediktion och Pinnacle closing: {len(joined):,}")
joined.head()

Testmatcher med prediktion och Pinnacle closing: 380


,H,D,A,Season,confidence,pick,market_H,market_D,market_A,price_H,price_D,price_A,result,League
Premier League_20230811_Burnley_Man City,0.0923,0.1825,0.7252,2023/24,0.7252,A,0.1011,0.1674,0.7314,9.6200,5.8100,1.3300,A,Premier League
Premier League_20230812_Arsenal_Nott'm Forest,0.7867,0.1612,0.0521,2023/24,0.7867,H,0.8176,0.1216,0.0608,1.1900,8.0000,16.0000,H,Premier League
Premier League_20230812_Bournemouth_West Ham,0.3424,0.2724,0.3852,2023/24,0.3852,A,0.3559,0.2719,0.3722,2.7500,3.6000,2.6300,D,Premier League
Premier League_20230812_Brighton_Luton,0.6265,0.2226,0.1509,2023/24,0.6265,H,0.7625,0.1523,0.0852,1.2700,6.3600,11.3600,H,Premier League
Premier League_20230812_Everton_Fulham,0.3965,0.2691,0.3344,2023/24,0.3965,H,0.4091,0.2945,0.2963,2.3900,3.3200,3.3000,A,Premier League


## 4. Prognoskvalitet på 2023/24

Samma matcher, tre prognoser. Modellen ska slå dummy. Closing-linjen är den svårare jämförelsen.

In [65]:
def macro_gini(y, probs, labels=OUTCOMES):
    scores = []
    for index, label in enumerate(labels):
        binary = y.eq(label).astype(int)
        if binary.nunique() < 2:
            continue
        scores.append(2 * roc_auc_score(binary, probs[:, index]) - 1)
    return float(np.mean(scores)) if scores else np.nan


def forecast_row(name, y, probs):
    predicted = np.array(OUTCOMES)[probs.argmax(axis=1)]
    return {
        "källa": name,
        "log_loss": log_loss(y, probs, labels=OUTCOMES),
        "accuracy": accuracy_score(y, predicted),
        "macro_gini": macro_gini(y, probs),
    }


y = joined["result"]
model_probs = joined[OUTCOMES].to_numpy(dtype=float)
market_probs = joined[[f"market_{o}" for o in OUTCOMES]].to_numpy(dtype=float)
priors = (
    raw.loc[raw["Season"].astype(str) < TEST_SEASON, "FullTimeResult"]
    .value_counts(normalize=True).reindex(OUTCOMES).fillna(0.0).to_numpy(dtype=float)
)
dummy_probs = np.tile(priors, (len(joined), 1))

quality = pd.DataFrame([
    forecast_row("Modell", y, model_probs),
    forecast_row("Pinnacle closing", y, market_probs),
    forecast_row("Konstanta träningspriors", y, dummy_probs),
]).set_index("källa")
quality

,log_loss,accuracy,macro_gini
källa,,,
Modell,1.5385,0.5868,0.3808
Pinnacle closing,1.5259,0.5974,0.4589
Konstanta träningspriors,1.1195,0.4605,0.0000


## 5. ROI på testsäsongen: modellens pick

1 enhet på det utfall modellen sätter högst sannolikhet på, settlat till Pinnacle closing. Det är ROI-motsvarigheten till accuracy i all-history-notebooken: vi tar modellens val, inte varje utfall med positiv EV.

In [66]:
def bet_predicted(frame):
    if frame.empty:
        return pd.DataFrame(columns=[
            "Season", "League", "outcome", "price", "model_prob", "market_prob",
            "confidence", "profit", "won",
        ])
    pick = frame["pick"] if "pick" in frame.columns else frame[OUTCOMES].idxmax(axis=1)
    confidence = frame["confidence"] if "confidence" in frame.columns else frame[OUTCOMES].max(axis=1)
    price_values = np.vstack([frame[f"price_{outcome}"] for outcome in OUTCOMES]).T
    pick_index = pick.map({outcome: index for index, outcome in enumerate(OUTCOMES)}).to_numpy()
    price = price_values[np.arange(len(frame)), pick_index]
    model_prob = frame[OUTCOMES].to_numpy(dtype=float)[np.arange(len(frame)), pick_index]
    market_prob = frame[[f"market_{outcome}" for outcome in OUTCOMES]].to_numpy(dtype=float)[
        np.arange(len(frame)), pick_index
    ]
    won = pick.eq(frame["result"])
    return pd.DataFrame({
        "Season": frame["Season"],
        "League": frame["League"],
        "outcome": pick,
        "price": price,
        "model_prob": model_prob,
        "market_prob": market_prob,
        "confidence": confidence,
        "profit": np.where(won, price - 1.0, -1.0),
        "won": won,
    })


def roi_row(bets, label):
    if bets.empty:
        return {
            "urval": label, "bets": 0, "win_rate": np.nan, "roi": np.nan,
            "roi_ci_low": np.nan, "roi_ci_high": np.nan, "profit_units": 0.0,
        }
    low, high = backtest.bootstrap_roi(bets["profit"].to_numpy())
    return {
        "urval": label,
        "bets": len(bets),
        "win_rate": float(bets["won"].mean()),
        "roi": float(bets["profit"].mean()),
        "roi_ci_low": low,
        "roi_ci_high": high,
        "profit_units": float(bets["profit"].sum()),
    }


pick_bets = bet_predicted(joined)
control = backtest.random_bet_control(joined)
print(f"Slumpmässigt utfall per match på {TEST_SEASON}: {100 * control:+.2f}% ROI")
pd.DataFrame([roi_row(pick_bets, f"Modellens pick, alla {TEST_SEASON}")])

Slumpmässigt utfall per match på 2023/24: -2.53% ROI


,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units
0,"Modellens pick, alla 2023/24",380,0.5868,0.0210,-0.0714,0.1153,7.9800


## 6. ROI bara när modellen är mer säker

Säkerhetsgränsen sätts på historisk walk-forward **före** testsäsongen, samma idé som median-OOF i all-history. Testeråret får inte välja tröskeln. Därefter visas också en svepning över fasta gränser, bara som beskrivning.

In [67]:
print(f"Walk-forward {CV_FIRST_SEASON}–säsongen före {TEST_SEASON} för OOF-säkerhet:")
oof_predictions = backtest.walk_forward_predictions(
    team_data.loc[team_data["Season"].astype(str) < TEST_SEASON],
    CV_FIRST_SEASON,
    verbose=True,
)
oof_predictions["confidence"] = oof_predictions[OUTCOMES].max(axis=1)
oof_joined = oof_predictions.join(market, how="inner")
confidence_threshold = float(oof_joined["confidence"].median())
print(f"\nOOF-matcher med odds: {len(oof_joined):,}")
print(f"Säkerhetsgräns (OOF-median): {confidence_threshold:.3f}")

confident = joined.loc[joined["confidence"] >= confidence_threshold]
confident_bets = bet_predicted(confident)

confident_quality = pd.DataFrame([
    forecast_row(f"Modell, {TEST_SEASON} alla", y, model_probs),
    forecast_row(
        f"Modell, {TEST_SEASON} säkra",
        confident["result"],
        confident[OUTCOMES].to_numpy(dtype=float),
    ) if len(confident) else {"källa": f"Modell, {TEST_SEASON} säkra", "log_loss": np.nan, "accuracy": np.nan, "macro_gini": np.nan},
]).set_index("källa")

roi_table = pd.DataFrame([
    roi_row(pick_bets, f"Pick, alla {TEST_SEASON}"),
    roi_row(confident_bets, f"Pick, säkra (≥ {confidence_threshold:.3f})"),
    roi_row(bet_predicted(oof_joined), "Pick, OOF 2018/19–2022/23"),
    roi_row(
        bet_predicted(oof_joined.loc[oof_joined["confidence"] >= confidence_threshold]),
        "Pick, OOF säkra",
    ),
])
print(f"Säkra testmatcher: {len(confident):,} / {len(joined):,} ({100 * len(confident) / len(joined):.1f}%)")
display(confident_quality)
roi_table

Walk-forward 2018/19–säsongen före 2023/24 för OOF-säkerhet:
  2018/19: trained on 13,680 team-rows, predicted 380 matches
  2019/20: trained on 14,440 team-rows, predicted 380 matches
  2020/21: trained on 15,200 team-rows, predicted 380 matches
  2021/22: trained on 15,960 team-rows, predicted 380 matches
  2022/23: trained on 16,720 team-rows, predicted 380 matches

OOF-matcher med odds: 1,900
Säkerhetsgräns (OOF-median): 0.526
Säkra testmatcher: 204 / 380 (53.7%)


,log_loss,accuracy,macro_gini
källa,,,
"Modell, 2023/24 alla",1.5385,0.5868,0.3808
"Modell, 2023/24 säkra",1.8754,0.6961,0.4869


,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units
0,"Pick, alla 2023/24",380,0.5868,0.0210,-0.0714,0.1153,7.9800
1,"Pick, säkra (≥ 0.526)",204,0.6961,0.0293,-0.0698,0.1271,5.9700
2,"Pick, OOF 2018/19–2022/23",1900,0.5405,-0.0190,-0.0637,0.0256,-36.0700
3,"Pick, OOF säkra",950,0.6568,-0.0091,-0.0568,0.0392,-8.6500


In [68]:
sweep = pd.DataFrame([
    roi_row(
        bet_predicted(joined.loc[joined["confidence"] >= threshold]),
        f"confidence ≥ {threshold:.2f}",
    )
    | {"threshold": threshold, "coverage": float((joined["confidence"] >= threshold).mean())}
    for threshold in CONFIDENCE_SWEEP
])
sweep

,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units,threshold,coverage
0,confidence ≥ 0.40,341,0.6012,0.0056,-0.0889,0.0984,1.9100,0.4000,0.8974
1,confidence ≥ 0.45,280,0.6179,-0.0259,-0.1211,0.0696,-7.2500,0.4500,0.7368
2,confidence ≥ 0.50,231,0.6667,0.0056,-0.0916,0.1016,1.3000,0.5000,0.6079
3,confidence ≥ 0.55,184,0.7065,0.0248,-0.0762,0.1257,4.5600,0.5500,0.4842
4,confidence ≥ 0.60,144,0.7431,0.0256,-0.0804,0.1265,3.6900,0.6000,0.3789
5,confidence ≥ 0.70,67,0.8060,0.0033,-0.1202,0.1172,0.2200,0.7000,0.1763


## 7. Säker, överens med marknaden, men mer övertygad

Tre villkor på samma pick:

1. Modellen är säker (OOF-medianen från förra steget).
2. Marknadens mest sannolika utfall är **samma** H, D eller A.
3. Modellens sannolikhet för det utfallet är **högre** än den deviggade Pinnacle-sannolikheten.

Det är inte samma sak som EV mot priset: juice kan göra ett vad negativt även när modellen ligger över linjen. Tabellen visar båda.

In [69]:
def with_agreement(frame):
    out = frame.copy()
    if "pick" not in out.columns:
        out["pick"] = out[OUTCOMES].idxmax(axis=1)
    if "confidence" not in out.columns:
        out["confidence"] = out[OUTCOMES].max(axis=1)
    market_cols = [f"market_{outcome}" for outcome in OUTCOMES]
    out["market_pick"] = out[market_cols].idxmax(axis=1).str.replace("market_", "", regex=False)
    pick_index = out["pick"].map({outcome: index for index, outcome in enumerate(OUTCOMES)}).to_numpy()
    out["model_pick_prob"] = out[OUTCOMES].to_numpy(dtype=float)[np.arange(len(out)), pick_index]
    out["market_pick_prob"] = out[market_cols].to_numpy(dtype=float)[np.arange(len(out)), pick_index]
    out["prob_edge"] = out["model_pick_prob"] - out["market_pick_prob"]
    out["agree"] = out["pick"].eq(out["market_pick"])
    out["model_more"] = out["prob_edge"] > 0
    return out


def select_agreed_value(frame, min_confidence, min_prob_edge=0.0):
    tagged = with_agreement(frame)
    return tagged.loc[
        (tagged["confidence"] >= min_confidence)
        & tagged["agree"]
        & (tagged["prob_edge"] > min_prob_edge)
    ]


tagged = with_agreement(joined)
oof_tagged = with_agreement(oof_joined)
agreed = tagged.loc[tagged["confidence"].ge(confidence_threshold) & tagged["agree"]]
agreed_value = select_agreed_value(joined, confidence_threshold)
oof_agreed_value = select_agreed_value(oof_joined, confidence_threshold)

print(
    f"Säkra: {int(tagged.confidence.ge(confidence_threshold).sum())}  |  "
    f"säkra + samma pick som marknaden: {len(agreed)}  |  "
    f"och modellen mer övertygad: {len(agreed_value)}"
)
print(
    f"OOF samma filter: {len(oof_agreed_value)} / {len(oof_tagged)} "
    f"({100 * len(oof_agreed_value) / len(oof_tagged):.1f}%)"
)

agreed_quality = pd.DataFrame([
    forecast_row(f"Modell, {TEST_SEASON} säkra", confident["result"], confident[OUTCOMES].to_numpy(dtype=float)),
    forecast_row(
        f"Modell, {TEST_SEASON} säkra + överens + mer övertygad",
        agreed_value["result"],
        agreed_value[OUTCOMES].to_numpy(dtype=float),
    ) if len(agreed_value) else {
        "källa": f"Modell, {TEST_SEASON} säkra + överens + mer övertygad",
        "log_loss": np.nan, "accuracy": np.nan, "macro_gini": np.nan,
    },
]).set_index("källa")

agreed_roi = pd.DataFrame([
    roi_row(confident_bets, f"Pick, säkra (≥ {confidence_threshold:.3f})"),
    roi_row(bet_predicted(agreed), "Pick, säkra + överens med marknaden"),
    roi_row(bet_predicted(agreed_value), "Pick, säkra + överens + modellen mer övertygad"),
    roi_row(bet_predicted(oof_agreed_value), "Samma filter, OOF 2018/19–2022/23"),
])
display(agreed_quality)
agreed_roi

Säkra: 204  |  säkra + samma pick som marknaden: 204  |  och modellen mer övertygad: 112
OOF samma filter: 554 / 1900 (29.2%)


,log_loss,accuracy,macro_gini
källa,,,
"Modell, 2023/24 säkra",1.8754,0.6961,0.4869
"Modell, 2023/24 säkra + överens + mer övertygad",1.8584,0.6607,0.4678


,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units
0,"Pick, säkra (≥ 0.526)",204,0.6961,0.0293,-0.0698,0.1271,5.9700
1,"Pick, säkra + överens med marknaden",204,0.6961,0.0293,-0.0698,0.1271,5.9700
2,"Pick, säkra + överens + modellen mer övertygad",112,0.6607,0.0505,-0.0990,0.1933,5.6600
3,"Samma filter, OOF 2018/19–2022/23",554,0.6155,-0.0081,-0.0774,0.0613,-4.4700


## 7b. Samma filter, OOF över tid

Walk-forward från 2013/14 (Pinnacle-eran) t.o.m. 2023/24. Varje säsong predikteras av en modell tränad bara på äldre säsonger.

Säkerhetsgränsen för säsong *S* är median-confidence i alla **tidigare** OOF-säsonger, inte i *S* själv. Första säsongen har ingen tidigare OOF och ingår därför inte i filtret. Därefter samma tre villkor: säker, samma pick som marknaden, modellen mer övertygad.

Fönstren räknas bakåt från 2023/24: senaste 3, senaste 7, och allt.

In [70]:
FIRST_OOF_SEASON = "2013/14"

print(f"Walk-forward {FIRST_OOF_SEASON}–{TEST_SEASON} (OOF, en modell per säsong):")
oof_all_predictions = backtest.walk_forward_predictions(team_data, FIRST_OOF_SEASON, verbose=True)
oof_all_predictions["confidence"] = oof_all_predictions[OUTCOMES].max(axis=1)
oof_all = with_agreement(oof_all_predictions.join(market, how="inner"))


def nested_confidence_threshold(frame):
    """Median-confidence per säsong, bara från äldre OOF-säsonger."""
    seasons = sorted(frame["Season"].astype(str).unique())
    thresholds = {}
    mask = pd.Series(False, index=frame.index)
    for index, season in enumerate(seasons):
        prior = frame.loc[frame["Season"].astype(str).isin(seasons[:index]), "confidence"]
        if prior.empty:
            thresholds[season] = np.nan
            continue
        threshold = float(prior.median())
        thresholds[season] = threshold
        this = frame["Season"].astype(str).eq(season)
        mask.loc[this] = frame.loc[this, "confidence"] >= threshold
    return mask, pd.Series(thresholds, name="threshold")


confident_mask, nested_thresholds = nested_confidence_threshold(oof_all)
oof_filter = oof_all.loc[confident_mask & oof_all["agree"] & oof_all["model_more"]]
oof_seasons = sorted(oof_all["Season"].astype(str).unique())
usable_seasons = [season for season, threshold in nested_thresholds.items() if pd.notna(threshold)]

windows = {
    "Senaste 3 säsongerna": usable_seasons[-3:],
    "Senaste 7 säsongerna": usable_seasons[-7:],
    "Alla OOF-säsonger": usable_seasons,
}

print("\nNestlad säkerhetsgräns per säsong (median av tidigare OOF):")
display(nested_thresholds.to_frame())

oof_window_rows = []
for label, seasons in windows.items():
    subset = oof_filter.loc[oof_filter["Season"].astype(str).isin(seasons)]
    row = roi_row(bet_predicted(subset), label)
    row["seasons"] = f"{seasons[0]}–{seasons[-1]}" if seasons else ""
    row["n_seasons"] = len(seasons)
    oof_window_rows.append(row)

oof_windows = pd.DataFrame(oof_window_rows)
print("Filter: OOF-säker + överens med marknaden + modellen mer övertygad")
display(oof_windows)

oof_by_season = pd.DataFrame([
    roi_row(bet_predicted(group), season)
    for season, group in oof_filter.groupby(oof_filter["Season"].astype(str), sort=True)
])
oof_by_season

Walk-forward 2013/14–2023/24 (OOF, en modell per säsong):
  2013/14: trained on 9,880 team-rows, predicted 380 matches
  2014/15: trained on 10,640 team-rows, predicted 380 matches
  2015/16: trained on 11,400 team-rows, predicted 380 matches
  2016/17: trained on 12,160 team-rows, predicted 380 matches
  2017/18: trained on 12,920 team-rows, predicted 380 matches
  2018/19: trained on 13,680 team-rows, predicted 380 matches
  2019/20: trained on 14,440 team-rows, predicted 380 matches
  2020/21: trained on 15,200 team-rows, predicted 380 matches
  2021/22: trained on 15,960 team-rows, predicted 380 matches
  2022/23: trained on 16,720 team-rows, predicted 380 matches
  2023/24: trained on 17,480 team-rows, predicted 380 matches

Nestlad säkerhetsgräns per säsong (median av tidigare OOF):


,threshold
2013/14,NaN
2014/15,0.5391
2015/16,0.5481
2016/17,0.5392
2017/18,0.5338
2018/19,0.5326
2019/20,0.5338
2020/21,0.5326
2021/22,0.5312
2022/23,0.5304


Filter: OOF-säker + överens med marknaden + modellen mer övertygad


,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units,seasons,n_seasons
0,Senaste 3 säsongerna,331,0.6314,0.0079,-0.0806,0.0948,2.6200,2021/22–2023/24,3
1,Senaste 7 säsongerna,734,0.6213,-0.0073,-0.0664,0.0501,-5.3600,2017/18–2023/24,7
2,Alla OOF-säsonger,1101,0.6158,-0.0022,-0.0516,0.0469,-2.4000,2014/15–2023/24,10


,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units
0,2014/15,184,0.6141,-0.0036,-0.1235,0.1144,-0.6600
1,2015/16,109,0.5688,-0.0477,-0.2073,0.1133,-5.2000
2,2016/17,74,0.6351,0.1192,-0.0851,0.3230,8.8200
3,2017/18,83,0.5663,-0.1064,-0.2795,0.0706,-8.8300
4,2018/19,109,0.6789,0.0800,-0.0724,0.2240,8.7200
5,2019/20,112,0.6429,0.0067,-0.1405,0.1531,0.7500
6,2020/21,99,0.5455,-0.0871,-0.2544,0.0853,-8.6200
7,2021/22,101,0.6436,0.0215,-0.1325,0.1745,2.1700
8,2022/23,119,0.5882,-0.0522,-0.2028,0.0891,-6.2100
9,2023/24,111,0.6667,0.0600,-0.0884,0.2086,6.6600


In [71]:
edge_sweep_rows = []
for min_prob_edge in (0.00, 0.01, 0.02, 0.03, 0.05, 0.08):
    subset = select_agreed_value(joined, confidence_threshold, min_prob_edge)
    row = roi_row(bet_predicted(subset), f"prob-edge > {min_prob_edge:.2f}")
    row["min_prob_edge"] = min_prob_edge
    row["coverage"] = float(len(subset) / len(joined))
    row["mean_prob_edge"] = float(subset["prob_edge"].mean()) if len(subset) else np.nan
    edge_sweep_rows.append(row)
pd.DataFrame(edge_sweep_rows)

,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units,min_prob_edge,coverage,mean_prob_edge
0,prob-edge > 0.00,112,0.6607,0.0505,-0.0990,0.1933,5.6600,0.0000,0.2947,0.0551
1,prob-edge > 0.01,97,0.6392,0.0360,-0.1302,0.1969,3.4900,0.0100,0.2553,0.0628
2,prob-edge > 0.02,86,0.6163,0.0269,-0.1506,0.2029,2.3100,0.0200,0.2263,0.0688
3,prob-edge > 0.03,74,0.5946,0.0019,-0.1934,0.1941,0.1400,0.0300,0.1947,0.0757
4,prob-edge > 0.05,50,0.5800,0.0416,-0.2118,0.2916,2.0800,0.0500,0.1316,0.0928
5,prob-edge > 0.08,24,0.5417,-0.0146,-0.3879,0.3558,-0.3500,0.0800,0.0632,0.1238


De valda vaden, sorterade efter hur mycket mer modellen tror än marknaden.

In [72]:
ids = raw.copy()
ids["MatchId"] = (
    ids["League"].astype(str) + "_"
    + pd.to_datetime(ids["MatchDate"]).dt.strftime("%Y%m%d") + "_"
    + ids["HomeTeam"].astype(str) + "_"
    + ids["AwayTeam"].astype(str)
)
lookup = ids.set_index("MatchId")[["MatchDate", "HomeTeam", "AwayTeam"]]

agreed_bets = bet_predicted(agreed_value).join(agreed_value[["market_pick", "prob_edge"]], how="left")
sample_agreed = agreed_bets.join(lookup, how="left").sort_values("prob_edge", ascending=False)
sample_agreed[[
    "MatchDate", "HomeTeam", "AwayTeam", "outcome", "market_pick", "price",
    "model_prob", "market_prob", "prob_edge", "confidence", "profit", "won",
]].head(20)

,MatchDate,HomeTeam,AwayTeam,outcome,market_pick,price,model_prob,market_prob,prob_edge,confidence,profit,won
Premier League_20231125_Newcastle_Chelsea,2023-11-25,Newcastle,Chelsea,H,H,2.6000,0.5741,0.3739,0.2002,0.5741,1.6000,True
Premier League_20231106_Tottenham_Chelsea,2023-11-06,Tottenham,Chelsea,H,H,2.3700,0.5860,0.4100,0.1761,0.5860,-1.0000,False
Premier League_20240131_Liverpool_Chelsea,2024-01-31,Liverpool,Chelsea,H,H,1.6600,0.7393,0.5842,0.1550,0.7393,0.6600,True
Premier League_20230923_Brentford_Everton,2023-09-23,Brentford,Everton,H,H,1.9500,0.6550,0.5011,0.1539,0.6550,-1.0000,False
Premier League_20231111_Bournemouth_Newcastle,2023-11-11,Bournemouth,Newcastle,A,A,1.9700,0.6461,0.4928,0.1532,0.6461,-1.0000,False
Premier League_20231231_Tottenham_Bournemouth,2023-12-31,Tottenham,Bournemouth,H,H,1.7900,0.6858,0.5415,0.1443,0.6858,0.7900,True
Premier League_20240217_Newcastle_Bournemouth,2024-02-17,Newcastle,Bournemouth,H,H,2.0000,0.6251,0.4861,0.1390,0.6251,-1.0000,False
Premier League_20230819_Man City_Newcastle,2023-08-19,Man City,Newcastle,H,H,1.8700,0.6508,0.5222,0.1286,0.6508,0.8700,True
Premier League_20231227_Brentford_Wolves,2023-12-27,Brentford,Wolves,H,H,2.2500,0.5588,0.4322,0.1266,0.5588,-1.0000,False
Premier League_20240402_Newcastle_Everton,2024-04-02,Newcastle,Everton,H,H,2.0300,0.6035,0.4801,0.1233,0.6035,-1.0000,False


## 8. Samma holdout, men EV mot marknaden

Här är vännens regel: 1 enhet på varje utfall vars förväntade värde vid closing klarar 0,05. Samma testsäsong, med och utan säkerhetsfilter. Om modellen är sämre än linjen plockar EV-filtret ofta de matcher där den har mest fel.

In [73]:
edge_all = backtest.simulate(joined, EDGE_THRESHOLD)
edge_confident = backtest.simulate(confident, EDGE_THRESHOLD)

edge_table = pd.DataFrame([
    roi_row(edge_all, f"EV > {EDGE_THRESHOLD:.2f}, alla {TEST_SEASON}"),
    roi_row(edge_confident, f"EV > {EDGE_THRESHOLD:.2f}, säkra"),
])
display(edge_table)

by_outcome = (
    pick_bets.groupby("outcome")
    .agg(bets=("profit", "size"), win_rate=("won", "mean"), roi=("profit", "mean"), profit_units=("profit", "sum"))
    .reindex(OUTCOMES)
)
print("Modellens pick per utfall, hela testsäsongen:")
by_outcome

,urval,bets,win_rate,roi,roi_ci_low,roi_ci_high,profit_units
0,"EV > 0.05, alla 2023/24",344,0.2326,-0.1792,-0.3602,0.0142,-61.6400
1,"EV > 0.05, säkra",190,0.2474,-0.1274,-0.3919,0.1593,-24.2100


Modellens pick per utfall, hela testsäsongen:


,bets,win_rate,roi,profit_units
outcome,,,,
H,251.0000,0.5976,0.0332,8.3300
D,NaN,NaN,NaN,NaN
A,129.0000,0.5659,-0.0027,-0.3500


## 9. De säkra vaden

Högst modellkonfidens bland de matcher som klarar OOF-gränsen, med faktiskt resultat och vinst/förlust.

In [74]:
ids = raw.copy()
ids["MatchId"] = (
    ids["League"].astype(str) + "_"
    + pd.to_datetime(ids["MatchDate"]).dt.strftime("%Y%m%d") + "_"
    + ids["HomeTeam"].astype(str) + "_"
    + ids["AwayTeam"].astype(str)
)
lookup = ids.set_index("MatchId")[["MatchDate", "HomeTeam", "AwayTeam"]]

sample = confident_bets.join(lookup, how="left").sort_values("confidence", ascending=False)
sample[[
    "MatchDate", "HomeTeam", "AwayTeam", "outcome", "price",
    "model_prob", "market_prob", "confidence", "profit", "won",
]].head(20)

,MatchDate,HomeTeam,AwayTeam,outcome,price,model_prob,market_prob,confidence,profit,won
Premier League_20240404_Liverpool_Sheffield United,2024-04-04,Liverpool,Sheffield United,H,1.0600,0.9108,0.9162,0.9108,0.0600,True
Premier League_20231230_Man City_Sheffield United,2023-12-30,Man City,Sheffield United,H,1.0700,0.9065,0.9040,0.9065,0.0700,True
Premier League_20240131_Man City_Burnley,2024-01-31,Man City,Burnley,H,1.1100,0.8867,0.8735,0.8867,0.1100,True
Premier League_20231028_Arsenal_Sheffield United,2023-10-28,Arsenal,Sheffield United,H,1.1300,0.8830,0.8527,0.8830,0.1300,True
Premier League_20230923_Man City_Nott'm Forest,2023-09-23,Man City,Nott'm Forest,H,1.1700,0.8625,0.8315,0.8625,0.1700,True
Premier League_20231104_Man City_Bournemouth,2023-11-04,Man City,Bournemouth,H,1.0900,0.8621,0.8826,0.8621,0.0900,True
Premier League_20240403_Arsenal_Luton,2024-04-03,Arsenal,Luton,H,1.1300,0.8615,0.8580,0.8615,0.1300,True
Premier League_20240519_Man City_West Ham,2024-05-19,Man City,West Ham,H,1.0700,0.8560,0.9011,0.8560,0.0700,True
Premier League_20240413_Man City_Luton,2024-04-13,Man City,Luton,H,1.0800,0.8535,0.8916,0.8535,0.0800,True
Premier League_20240210_Liverpool_Burnley,2024-02-10,Liverpool,Burnley,H,1.1600,0.8515,0.8330,0.8515,0.1600,True


## 10. Closing-line value och edge mot opening

Spelet läggs till **högsta tillgängliga odds** i football-data (`MaxH`/`MaxD`/`MaxA`): det bolag som ger högst faktor på just det utfallet, till exempel Bet365 eller Pinnacle. Betsson finns inte som egen kolumn i den här källan.

**Edge** jämförs mot Pinnacles *deviggade* opening-linje (samma bolag, tre utfall). Max-trion ska inte deviggas: de tre talen kommer ofta från tre olika böcker.

**CLV** mäts mot Pinnacle closing, den skarpa stängningen:

$$CLV = \frac{\text{bästa opening}}{\text{Pinnacle closing}} - 1$$

ROI settlas till det högsta oddset, alltså priset ni faktiskt hade fått. Trösklar väljs på OOF före 2023/24 efter CLV, inte efter högst ROI.


In [75]:
from src.team_dataset import match_id as make_match_id

OPEN_MAX = ["MaxH", "MaxD", "MaxA"]
B365_OPEN = ["B365H", "B365D", "B365A"]
EDGE_BINS = [-np.inf, 0.0, 0.02, 0.05, 0.10, np.inf]
EDGE_LABELS = ["Under 0 %", "0–2 %", "2–5 %", "5–10 %", "Över 10 %"]
MIN_EDGE_CANDIDATES = (0.02, 0.05, 0.08)
PRE_REGISTERED_EDGE = 0.05
MAX_ODDS_CAP = 5.0


def line_table(raw_matches, columns, prefix):
    """Komplett 1X2 från samma källa, med deviggade sannolikheter."""
    usable = raw_matches.dropna(subset=list(columns)).copy()
    usable = usable.loc[(usable[list(columns)] > 1).all(axis=1)]
    if usable.empty:
        raise ValueError(f"Inga rader med giltiga odds för {list(columns)}")
    index = make_match_id(usable)
    prices = pd.DataFrame(
        usable[list(columns)].to_numpy(dtype=float),
        columns=OUTCOMES,
        index=index,
    )
    prices = prices.loc[~prices.index.duplicated()]
    market = pd.DataFrame(backtest.devig(prices), columns=OUTCOMES, index=prices.index)
    return prices.add_prefix(f"{prefix}_price_").join(market.add_prefix(f"{prefix}_market_"))


def prices_only(raw_matches, columns, prefix):
    """En kolumn per utfall, utan krav på komplett trio. Ogiltiga odds blir NA."""
    present = [name for name in columns if name in raw_matches.columns]
    if len(present) != 3:
        return pd.DataFrame()
    numeric = raw_matches[present].apply(pd.to_numeric, errors="coerce")
    numeric = numeric.where(numeric > 1)
    index = make_match_id(raw_matches)
    prices = pd.DataFrame(numeric.to_numpy(dtype=float), columns=OUTCOMES, index=index)
    prices = prices.loc[~prices.index.duplicated()]
    return prices.add_prefix(f"{prefix}_price_")


def pick_column(frame, columns):
    index = frame["pick"].map({outcome: i for i, outcome in enumerate(OUTCOMES)}).to_numpy()
    return frame[columns].to_numpy(dtype=float)[np.arange(len(frame)), index]


def clv_summary(bets, label):
    if bets.empty:
        return {
            "urval": label, "bets": 0, "win_rate": np.nan, "roi": np.nan,
            "mean_clv": np.nan, "share_clv_pos": np.nan, "mean_edge": np.nan,
            "mean_open_odds": np.nan, "mean_close_odds": np.nan,
        }
    return {
        "urval": label,
        "bets": len(bets),
        "win_rate": float(bets["won"].mean()),
        "roi": float(bets["profit"].mean()),
        "mean_clv": float(bets["clv"].mean()),
        "share_clv_pos": float((bets["clv"] > 0).mean()),
        "mean_edge": float(bets["edge"].mean()),
        "mean_open_odds": float(bets["open_odds"].mean()),
        "mean_close_odds": float(bets["close_odds"].mean()),
    }


if "oof_all_predictions" not in globals():
    FIRST_OOF_SEASON = "2013/14"
    print(f"Walk-forward saknas, kör {FIRST_OOF_SEASON}–{TEST_SEASON} …")
    oof_all_predictions = backtest.walk_forward_predictions(team_data, FIRST_OOF_SEASON, verbose=True)

preds = oof_all_predictions.copy()
preds["pick"] = preds[OUTCOMES].idxmax(axis=1)
preds["confidence"] = preds[OUTCOMES].max(axis=1)

open_ps = line_table(raw, backtest.OPENING_ODDS, "ps")
close_ps = line_table(raw, backtest.CLOSING_ODDS, "close")
open_max = prices_only(raw, OPEN_MAX, "max")
open_b365 = prices_only(raw, B365_OPEN, "b365")

meta = raw.copy()
meta["MatchId"] = make_match_id(meta)
meta = meta.set_index("MatchId")[["MatchDate", "KickoffTime", "HomeTeam", "AwayTeam"]]
meta = meta.loc[~meta.index.duplicated()]

ledger = (
    preds.join(open_ps, how="inner")
    .join(close_ps, how="inner")
    .join(open_max, how="left")
    .join(open_b365, how="left")
    .join(meta, how="left")
)
results = backtest.build_market(raw, odds_columns=backtest.CLOSING_ODDS)["result"]
ledger = ledger.join(results.rename("result"), how="inner")

ledger["ps_odds"] = pick_column(ledger, [f"ps_price_{o}" for o in OUTCOMES])
ledger["b365_odds"] = pick_column(ledger, [f"b365_price_{o}" for o in OUTCOMES]) if "b365_price_H" in ledger.columns else np.nan
ledger["max_odds"] = pick_column(ledger, [f"max_price_{o}" for o in OUTCOMES]) if "max_price_H" in ledger.columns else np.nan
ledger["close_odds"] = pick_column(ledger, [f"close_price_{o}" for o in OUTCOMES])
ledger["model_prob"] = pick_column(ledger, OUTCOMES)
ledger["open_market_prob"] = pick_column(ledger, [f"ps_market_{o}" for o in OUTCOMES])
ledger["close_market_prob"] = pick_column(ledger, [f"close_market_{o}" for o in OUTCOMES])

# Högsta faktorn bland Max, Bet365 och Pinnacle på modellens pick.
book_odds = pd.DataFrame({
    "Max": ledger["max_odds"],
    "Bet365": ledger["b365_odds"],
    "Pinnacle": ledger["ps_odds"],
})
ledger["open_odds"] = book_odds.max(axis=1, skipna=True)
ledger["bookmaker"] = book_odds.idxmax(axis=1)
ledger["placement"] = "best available opening"
missing_price = ledger["open_odds"].isna() | (ledger["open_odds"] <= 1)
ledger = ledger.loc[~missing_price].copy()

ledger["edge"] = ledger["model_prob"] - ledger["open_market_prob"]
ledger["price_ev"] = ledger["model_prob"] * ledger["open_odds"] - 1.0
ledger["clv"] = ledger["open_odds"] / ledger["close_odds"] - 1.0
ledger["won"] = ledger["pick"].eq(ledger["result"])
ledger["profit"] = np.where(ledger["won"], ledger["open_odds"] - 1.0, -1.0)
ledger["edge_bucket"] = pd.cut(ledger["edge"], bins=EDGE_BINS, labels=EDGE_LABELS, right=False)

oof_dev = ledger.loc[ledger["Season"].astype(str) < TEST_SEASON].copy()
holdout = ledger.loc[ledger["Season"].astype(str).eq(TEST_SEASON)].copy()

ledger_path = root / "data" / "clv_ledger_premier_league.csv"
ledger_path.parent.mkdir(exist_ok=True)
save_cols = [
    "Season", "MatchDate", "KickoffTime", "HomeTeam", "AwayTeam", "bookmaker",
    "placement", "pick", "result", "model_prob", "open_odds", "close_odds",
    "ps_odds", "b365_odds", "max_odds", "open_market_prob", "close_market_prob",
    "edge", "price_ev", "clv", "profit", "won",
]
ledger[save_cols].to_csv(ledger_path)
print("Spelet läggs till högsta opening bland Max / Bet365 / Pinnacle.")
print(ledger["bookmaker"].value_counts().to_string())
print(f"\nLedger: {len(ledger):,} matcher")
print(f"OOF före {TEST_SEASON}: {len(oof_dev):,}  |  holdout {TEST_SEASON}: {len(holdout):,}")
print(f"Snitt extra vs Pinnacle opening: {100 * (ledger['open_odds'] / ledger['ps_odds'] - 1).mean():+.2f}%")
print(f"Sparat: {ledger_path}")
ledger[save_cols].head()


Spelet läggs till högsta opening bland Max / Bet365 / Pinnacle.
Max         4170
Pinnacle       9
Bet365         1

Ledger: 4,180 matcher
OOF före 2023/24: 3,800  |  holdout 2023/24: 380
Snitt extra vs Pinnacle opening: +2.32%
Sparat: /Users/andreasnilsson/Desktop/Repos/betting-optimizer/data/clv_ledger_premier_league.csv


,Season,MatchDate,KickoffTime,HomeTeam,AwayTeam,bookmaker,placement,pick,result,model_prob,open_odds,close_odds,ps_odds,b365_odds,max_odds,open_market_prob,close_market_prob,edge,price_ev,clv,profit,won
Premier League_20130817_Arsenal_Aston Villa,2013/14,2013-08-17,NaN,Arsenal,Aston Villa,Max,best available opening,H,A,0.7501,1.4400,1.4400,1.4100,1.4400,1.4400,0.6939,0.6817,0.0562,0.0802,0.0000,-1.0000,False
Premier League_20130817_Liverpool_Stoke,2013/14,2013-08-17,NaN,Liverpool,Stoke,Max,best available opening,H,H,0.7167,1.4400,1.4200,1.4100,1.4000,1.4400,0.6946,0.6912,0.0221,0.0320,0.0141,0.4400,True
Premier League_20130817_Norwich_Everton,2013/14,2013-08-17,NaN,Norwich,Everton,Max,best available opening,A,D,0.4588,2.4000,2.2100,2.3500,2.4000,2.4000,0.4172,0.4433,0.0417,0.1012,0.0860,-1.0000,False
Premier League_20130817_Sunderland_Fulham,2013/14,2013-08-17,NaN,Sunderland,Fulham,Max,best available opening,H,A,0.4145,2.3000,2.5200,2.2500,2.3000,2.3000,0.4355,0.3879,-0.0210,-0.0467,-0.0873,-1.0000,False
Premier League_20130817_Swansea_Man United,2013/14,2013-08-17,NaN,Swansea,Man United,Max,best available opening,A,A,0.6074,2.1000,2.2200,2.0300,2.0000,2.1000,0.4827,0.4416,0.1247,0.2755,-0.0541,1.1000,True


In [76]:
def bucket_table(frame, title):
    rows = []
    for label in EDGE_LABELS:
        rows.append(clv_summary(frame.loc[frame["edge_bucket"] == label], label))
    rows.append(clv_summary(frame, "Alla picks"))
    table = pd.DataFrame(rows)
    print(title)
    return table


print("Viktigaste testet: edge ≥ 5 % på opening → blev closing lägre?")
key_oof = oof_dev.loc[oof_dev["edge"] >= PRE_REGISTERED_EDGE]
key_test = holdout.loc[holdout["edge"] >= PRE_REGISTERED_EDGE]
key_table = pd.DataFrame([
    clv_summary(key_oof, f"OOF, edge ≥ {PRE_REGISTERED_EDGE:.0%}"),
    clv_summary(key_test, f"{TEST_SEASON}, edge ≥ {PRE_REGISTERED_EDGE:.0%}"),
])
display(key_table)
if len(key_oof) and len(key_test):
    print(
        "Andel där closing kortats (CLV > 0): "
        f"OOF {100 * key_oof['clv'].gt(0).mean():.1f}%  |  "
        f"{TEST_SEASON} {100 * key_test['clv'].gt(0).mean():.1f}%"
    )
else:
    print("För få matcher i 5 %-testet.")

display(bucket_table(oof_dev, f"\nEdge-intervall, OOF före {TEST_SEASON} (tröskel väljs här):"))
bucket_table(holdout, f"\nEdge-intervall, orörd {TEST_SEASON}:")

Viktigaste testet: edge ≥ 5 % på opening → blev closing lägre?


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,"OOF, edge ≥ 5%",1208,0.4578,-0.0041,0.0180,0.6109,0.0940,2.3733,2.3440
1,"2023/24, edge ≥ 5%",107,0.4206,-0.0315,0.0206,0.6075,0.0934,2.4143,2.3764


Andel där closing kortats (CLV > 0): OOF 61.1%  |  2023/24 60.7%

Edge-intervall, OOF före 2023/24 (tröskel väljs här):


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,Under 0 %,1448,0.6202,0.0404,0.0230,0.6747,-0.0482,1.7603,1.7254
1,0–2 %,476,0.5462,-0.0459,0.0232,0.7080,0.0100,1.8854,1.8484
2,2–5 %,668,0.5314,0.0053,0.0198,0.6078,0.0346,2.0079,1.9767
3,5–10 %,773,0.4787,0.0035,0.0192,0.6352,0.0721,2.2663,2.2356
4,Över 10 %,435,0.4207,-0.0176,0.0158,0.5678,0.1328,2.5635,2.5366
5,Alla picks,3800,0.5437,0.0093,0.0209,0.6468,0.0188,2.0144,1.9816



Edge-intervall, orörd 2023/24:


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,Under 0 %,170,0.6588,0.0828,0.0275,0.7000,-0.0463,1.7265,1.6889
1,0–2 %,50,0.6600,0.1522,0.0230,0.6400,0.0114,1.8292,1.7928
2,2–5 %,53,0.6226,0.0649,0.0374,0.7170,0.0329,1.8353,1.7681
3,5–10 %,69,0.4638,0.0423,0.0164,0.6087,0.0726,2.2613,2.2399
4,Över 10 %,38,0.3421,-0.1655,0.0283,0.6053,0.1312,2.6921,2.6242
5,Alla picks,380,0.5868,0.0572,0.0263,0.6684,0.0116,1.9488,1.9072


In [77]:
candidate_rows = []
for minimum in MIN_EDGE_CANDIDATES:
    subset = oof_dev.loc[oof_dev["edge"] >= minimum]
    row = clv_summary(subset, f"OOF, min edge {minimum:.0%}")
    row["min_edge"] = minimum
    candidate_rows.append(row)
candidates = pd.DataFrame(candidate_rows)

# Välj på CLV, inte ROI. Vid lika CLV: högre tröskel (mer konservativt).
chosen_edge = float(
    candidates.sort_values(["mean_clv", "min_edge"], ascending=[False, False])["min_edge"].iloc[0]
) if candidates["bets"].gt(0).any() else PRE_REGISTERED_EDGE

print("Kandidater på OOF (urval efter CLV, inte ROI):")
display(candidates)
print(f"OOF-vald min-edge: {chosen_edge:.0%}. Utvärderas orörd på {TEST_SEASON}.")

holdout_eval = pd.DataFrame([
    clv_summary(holdout, f"{TEST_SEASON}, alla picks"),
    clv_summary(holdout.loc[holdout["edge"] >= PRE_REGISTERED_EDGE], f"{TEST_SEASON}, förregistrerat ≥ 5 %"),
    clv_summary(holdout.loc[holdout["edge"] >= chosen_edge], f"{TEST_SEASON}, OOF-vald ≥ {chosen_edge:.0%}"),
    clv_summary(
        holdout.loc[(holdout["edge"] >= chosen_edge) & (holdout["clv"] > 0)],
        f"{TEST_SEASON}, OOF-vald och denna match hade +CLV (diagnostik, inte en live-regel)",
    ),
])
display(holdout_eval)

print("\nPer utfall, OOF, edge ≥ 5 %:")
by_outcome_oof = pd.DataFrame([
    clv_summary(key_oof.loc[key_oof["pick"] == outcome], outcome)
    for outcome in OUTCOMES
])
display(by_outcome_oof)

odds_bins = [1.0, 1.70, 2.50, 4.00, np.inf]
odds_labels = ["1.00–1.70", "1.70–2.50", "2.50–4.00", "≥ 4.00"]
oof_dev = oof_dev.copy()
oof_dev["odds_band"] = pd.cut(oof_dev["open_odds"], bins=odds_bins, labels=odds_labels, right=False)
print("\nPer oddsintervall, OOF, edge ≥ 5 %:")
display(pd.DataFrame([
    clv_summary(key_oof.loc[pd.cut(key_oof["open_odds"], bins=odds_bins, labels=odds_labels, right=False) == band], band)
    for band in odds_labels
]))

capped = oof_dev.loc[(oof_dev["edge"] >= PRE_REGISTERED_EDGE) & (oof_dev["open_odds"] < MAX_ODDS_CAP)]
capped_test = holdout.loc[(holdout["edge"] >= PRE_REGISTERED_EDGE) & (holdout["open_odds"] < MAX_ODDS_CAP)]
print(f"\nUndvik höga odds (opening < {MAX_ODDS_CAP:.1f}) vid edge ≥ 5 %:")
display(pd.DataFrame([
    clv_summary(capped, f"OOF, edge ≥ 5 % och odds < {MAX_ODDS_CAP:.0f}"),
    clv_summary(capped_test, f"{TEST_SEASON}, edge ≥ 5 % och odds < {MAX_ODDS_CAP:.0f}"),
]))

Kandidater på OOF (urval efter CLV, inte ROI):


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds,min_edge
0,"OOF, min edge 2%",1876,0.4840,-0.0007,0.0186,0.6098,0.0728,2.2432,2.2132,0.0200
1,"OOF, min edge 5%",1208,0.4578,-0.0041,0.0180,0.6109,0.0940,2.3733,2.3440,0.0500
2,"OOF, min edge 8%",674,0.4421,0.0159,0.0135,0.5742,0.1176,2.5101,2.4893,0.0800


OOF-vald min-edge: 2%. Utvärderas orörd på 2023/24.


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,"2023/24, alla picks",380,0.5868,0.0572,0.0263,0.6684,0.0116,1.9488,1.9072
1,"2023/24, förregistrerat ≥ 5 %",107,0.4206,-0.0315,0.0206,0.6075,0.0934,2.4143,2.3764
2,"2023/24, OOF-vald ≥ 2%",160,0.4875,0.0004,0.0262,0.6438,0.0733,2.2225,2.1749
3,"2023/24, OOF-vald och denna match hade +CLV (d...",103,0.5340,0.1416,0.0663,1.0000,0.0723,2.2021,2.0586



Per utfall, OOF, edge ≥ 5 %:


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,H,915,0.4689,0.0021,0.0152,0.6055,0.0952,2.3238,2.3006
1,D,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A,293,0.4232,-0.0234,0.0267,0.6280,0.0903,2.5281,2.4792



Per oddsintervall, OOF, edge ≥ 5 %:


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,1.00–1.70,205,0.7073,0.0473,0.0215,0.6976,0.0823,1.4858,1.4572
1,1.70–2.50,501,0.4870,0.0141,0.0218,0.6287,0.0917,2.1096,2.0727
2,2.50–4.00,485,0.3278,-0.0520,0.0106,0.5546,0.0992,2.9518,2.9375
3,≥ 4.00,17,0.2941,0.2076,0.0704,0.6471,0.1535,4.3482,4.0988



Undvik höga odds (opening < 5.0) vid edge ≥ 5 %:


,urval,bets,win_rate,roi,mean_clv,share_clv_pos,mean_edge,mean_open_odds,mean_close_odds
0,"OOF, edge ≥ 5 % och odds < 5",1205,0.4589,-0.0016,0.0176,0.6100,0.0938,2.3666,2.3389
1,"2023/24, edge ≥ 5 % och odds < 5",106,0.4245,-0.0224,0.0202,0.6038,0.0920,2.3899,2.3547


In [78]:
print("Vilket bolag gav högst faktor på modellens pick?")
display(ledger["bookmaker"].value_counts().rename("matcher").to_frame())

print("\nMedelodds på samma pick:")
display(ledger[["ps_odds", "b365_odds", "max_odds", "open_odds"]].mean().rename("medelodds").to_frame())

print(
    "\nTolkning: ROI är nu till bästa pris. Edge är fortfarande mot Pinnacles "
    "deviggade opening. CLV är bästa opening mot Pinnacle closing. "
    "Betsson finns inte i football-data; Max är högsta noterade oddset bland "
    "böckerna i filen (ofta Bet365, Pinnacle, William Hill m.fl.)."
)
ledger.loc[ledger["edge"] >= PRE_REGISTERED_EDGE, [
    "MatchDate", "HomeTeam", "AwayTeam", "pick", "bookmaker", "model_prob",
    "ps_odds", "b365_odds", "open_odds", "close_odds", "edge", "clv", "profit", "won", "Season",
]].sort_values("edge", ascending=False).head(15)


Vilket bolag gav högst faktor på modellens pick?


,matcher
Max,4170
Pinnacle,9
Bet365,1



Medelodds på samma pick:


,medelodds
ps_odds,1.9628
b365_odds,1.9345
max_odds,2.0084
open_odds,2.0084



Tolkning: ROI är nu till bästa pris. Edge är fortfarande mot Pinnacles deviggade opening. CLV är bästa opening mot Pinnacle closing. Betsson finns inte i football-data; Max är högsta noterade oddset bland böckerna i filen (ofta Bet365, Pinnacle, William Hill m.fl.).


,MatchDate,HomeTeam,AwayTeam,pick,bookmaker,model_prob,ps_odds,b365_odds,open_odds,close_odds,edge,clv,profit,won,Season
Premier League_20200726_West Ham_Aston Villa,2020-07-26,West Ham,Aston Villa,H,Max,0.6028,3.1200,3.0000,3.2000,2.6600,0.2915,0.2030,-1.0000,False,2019/20
Premier League_20170521_Man United_Crystal Palace,2017-05-21,Man United,Crystal Palace,H,Max,0.6870,2.4400,2.5000,2.6000,2.3500,0.2862,0.1064,1.6000,True,2016/17
Premier League_20200722_Liverpool_Chelsea,2020-07-22,Liverpool,Chelsea,H,Max,0.7441,2.0800,2.0000,2.1300,1.9300,0.2768,0.1036,1.1300,True,2019/20
Premier League_20230901_Luton_West Ham,2023-09-01,Luton,West Ham,H,Max,0.4432,4.8200,4.7500,5.0000,4.6700,0.2411,0.0707,-1.0000,False,2023/24
Premier League_20200927_Sheffield United_Leeds,2020-09-27,Sheffield United,Leeds,H,Max,0.5696,2.8300,2.7000,2.8300,3.0900,0.2230,-0.0841,-1.0000,False,2020/21
Premier League_20200624_Newcastle_Aston Villa,2020-06-24,Newcastle,Aston Villa,H,Max,0.6355,2.3500,2.3000,2.4200,2.4000,0.2228,0.0083,-1.0000,False,2019/20
Premier League_20161030_Southampton_Chelsea,2016-10-30,Southampton,Chelsea,H,Max,0.5190,3.2000,3.2000,3.3000,3.2400,0.2129,0.0185,-1.0000,False,2016/17
Premier League_20210511_Man United_Leicester,2021-05-11,Man United,Leicester,H,Max,0.5910,2.5900,2.5000,2.6400,3.9400,0.2129,-0.3299,-1.0000,False,2020/21
Premier League_20150830_Southampton_Norwich,2015-08-30,Southampton,Norwich,H,Max,0.6885,2.0500,2.0500,2.0700,2.1100,0.2104,-0.0190,1.0700,True,2015/16
Premier League_20231125_Newcastle_Chelsea,2023-11-25,Newcastle,Chelsea,H,Max,0.5741,2.6500,2.5500,2.7200,2.6000,0.2087,0.0462,1.7200,True,2023/24


## 11. Samma urval, settlat till bästa bolag

Avsnitt 5–7 ovan är oförändrade: samma matcher, samma filter, **Pinnacle closing**. Därför ändrades inte 204 / 112 / 554 när vi la till fler bolag.

Här är **samma filter** (säker, överens med Pinnacle, modellen mer övertygad), men vinsten räknas till det **högsta opening-oddset** bland namngivna böcker. Win rate ska vara densamma; ROI och profit ska skilja sig om något bolag betalade mer än Pinnacle.

Först: vilka bolag *hade* de bästa opening-priserna på 2023/24? Andelen är av alla 1X2-utfall (H, D och A på varje match), inte av modellens pick.

In [ ]:
import importlib
from src import football_data as football_data_mod
from src.team_dataset import match_id as make_match_id

importlib.reload(football_data_mod)
from src.football_data import best_named_opening, bookmaker_label, opening_book_codes

if "WHH" not in raw.columns and "BWH" not in raw.columns:
    print("Parsar om matcher från cache så alla bolag följer med …")
    raw_all = football_data_mod.load_matches(divisions=["E0"])
    raw = raw_all.loc[
        (raw_all["League"] == LEAGUE) & (raw_all["Season"].astype(str) <= TEST_SEASON)
    ].copy()
    raw["MatchDate"] = pd.to_datetime(raw["MatchDate"])

test_raw = raw.loc[raw["Season"].astype(str).eq(TEST_SEASON)].copy()
best_test = best_named_opening(test_raw)
n_prices = int(best_test[[f"best_book_{o}" for o in OUTCOMES]].notna().to_numpy().sum())
book_hits = pd.concat(
    [best_test[f"best_book_{o}"].dropna() for o in OUTCOMES],
    ignore_index=True,
)
share = book_hits.value_counts()
book_share = pd.DataFrame({
    "bolag": [bookmaker_label(code) for code in share.index],
    "bästa_priser": share.to_numpy(),
    "andel": share.to_numpy() / n_prices,
}).head(10)
book_share.index = range(1, len(book_share) + 1)

print(
    f"{TEST_SEASON}: {len(test_raw)} matcher, {n_prices} opening-priser med minst ett namngivet bolag."
)
print("Top 10 bolag efter andel av högsta opening-oddset (H, D och A):")
display(book_share)

best_all = best_named_opening(raw)
best_all = best_all.copy()
best_all.index = make_match_id(raw)
best_all = best_all.loc[~best_all.index.duplicated()]


def settle_at_best(frame):
    """Samma rader och filter, men price_* är högsta namngivna opening."""
    out = frame.copy()
    aligned = best_all.reindex(out.index)
    for outcome in OUTCOMES:
        out[f"price_{outcome}"] = aligned[f"best_price_{outcome}"]
        out[f"best_book_{outcome}"] = aligned[f"best_book_{outcome}"]
    pick = out["pick"] if "pick" in out.columns else out[OUTCOMES].idxmax(axis=1)
    pick_index = pick.map({outcome: i for i, outcome in enumerate(OUTCOMES)}).to_numpy()
    pick_price = np.vstack([out[f"price_{o}"] for o in OUTCOMES]).T[
        np.arange(len(out)), pick_index
    ]
    return out.loc[pd.Series(pick_price, index=out.index).gt(1)]


joined_best = settle_at_best(joined)
oof_joined_best = settle_at_best(oof_joined)
tagged_best = with_agreement(joined_best)
confident_best = joined_best.loc[joined_best["confidence"] >= confidence_threshold]
agreed_best = tagged_best.loc[
    tagged_best["confidence"].ge(confidence_threshold) & tagged_best["agree"]
]
agreed_value_best = select_agreed_value(joined_best, confidence_threshold)
oof_agreed_value_best = select_agreed_value(oof_joined_best, confidence_threshold)

pick_book = []
for outcome in OUTCOMES:
    mask = joined_best["pick"].eq(outcome)
    pick_book.append(joined_best.loc[mask, f"best_book_{outcome}"])
pick_share = pd.concat(pick_book).value_counts(normalize=True).head(10)
pick_share_table = pd.DataFrame({
    "bolag": [bookmaker_label(code) for code in pick_share.index],
    "andel_av_modellens_pick": pick_share.to_numpy(),
})
print(f"\nSamma säsong, men bara modellens pick ({len(joined_best)} matcher):")
display(pick_share_table)

best_roi = pd.DataFrame([
    roi_row(
        bet_predicted(confident_best),
        f"Pick, säkra (≥ {confidence_threshold:.3f})",
    ),
    roi_row(bet_predicted(agreed_best), "Pick, säkra + överens med marknaden"),
    roi_row(
        bet_predicted(agreed_value_best),
        "Pick, säkra + överens + modellen mer övertygad",
    ),
    roi_row(
        bet_predicted(oof_agreed_value_best),
        "Samma filter, OOF 2018/19–2022/23",
    ),
])
print(
    "Samma filter som avsnitt 7, settlat till bästa namngivna opening "
    f"(Pinnacle-tabellen hade {len(confident_bets)} / {len(agreed)} / "
    f"{len(agreed_value)} / {len(oof_agreed_value)} bets):"
)
best_roi